# NeoOLAF DocRED Run and Controlled Evaluation KG Export

This notebook runs the existing NeoOLAF benchmark runner without modifying
NeoOLAF source code. It then projects each native `kg_local.json` and
`kg_inferred.json` into evaluation files constrained by a fixed relation
vocabulary.

NeoOLAF may still discover and add new relations to its generated ontology.
Those relations remain in the native artifacts but are excluded from the
evaluation KG unless they are permitted by the selected reference
ontology/list mode.

In [25]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
from pathlib import Path


def find_project_root() -> Path:
    # Support launching from the repository root or examples/RAGTreeDatasets.
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / "experiments/methods/run_neoolaf.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find experiments/methods/run_neoolaf.py. "
        "Launch this notebook from inside the NeoOLAF/RAGTree repository."
    )


PROJECT_ROOT = find_project_root()
RUNNER_PATH = PROJECT_ROOT / "experiments/methods/run_neoolaf.py"
PROJECTOR_PATH = PROJECT_ROOT / "experiments/methods/evaluation_kg_projection.py"
OFFLINE_RUNTIME_DIR = PROJECT_ROOT / "experiments/methods/neoolaf_offline_runtime"

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"RUNNER_PATH={RUNNER_PATH}")
print(f"PROJECTOR_PATH={PROJECTOR_PATH}")
print(f"OFFLINE_RUNTIME_DIR={OFFLINE_RUNTIME_DIR}")

PROJECT_ROOT=/home/galencarmedeiro/git/postdoc/NeoOLAF
RUNNER_PATH=/home/galencarmedeiro/git/postdoc/NeoOLAF/experiments/methods/run_neoolaf.py
PROJECTOR_PATH=/home/galencarmedeiro/git/postdoc/NeoOLAF/experiments/methods/evaluation_kg_projection.py
OFFLINE_RUNTIME_DIR=/home/galencarmedeiro/git/postdoc/NeoOLAF/experiments/methods/neoolaf_offline_runtime


## Configuration

`RELATION_FILTER_MODE` accepts:

- `list-only`, recommended for DocRED
- `ontology-only`
- `union`
- `intersection`

`REFERENCE_ONTOLOGY_PATH` is used only by the evaluation projector. It
must be a fixed dataset/reference ontology, not the ontology generated by
the current NeoOLAF run.

In [26]:
# Dataset and NeoOLAF inputs.
# This cell intentionally resolves the same RAGTree paths used before:
#   ../../../ragtree/data/preprocessed/docred_causal.jsonl
#   ../../../ragtree/data/ontology/DocREDOntology/ontology.ttl
# relative to examples/RAGTreeDatasets, but in a robust way.

NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"

def first_existing_path(label: str, candidates: list[Path]) -> Path:
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        candidate = candidate if candidate.is_absolute() else (PROJECT_ROOT / candidate)
        candidate = candidate.resolve()
        checked.append(candidate)
        if candidate.is_file():
            print(f"{label}={candidate}")
            return candidate
    message = [f"Could not find {label}. Tried:"]
    message.extend(f"  - {path}" for path in checked)
    message.append("Set the variable manually in this cell if your local layout is different.")
    raise FileNotFoundError("\n".join(message))

# The first candidates reproduce the working paths from the earlier notebook.
DATASET_JSONL = first_existing_path(
    "DATASET_JSONL",
    [
        NOTEBOOK_DIR / "../../../ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "RAGTree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT / "ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT / "data/RAGTreeDatasets/docred_causal.jsonl",
        PROJECT_ROOT / "data/RAGTreeDatasets/docred/dev.jsonl",
    ],
)

NEOOLAF_ONTOLOGY_PATH = first_existing_path(
    "NEOOLAF_ONTOLOGY_PATH",
    [
        NOTEBOOK_DIR / "../../../ragtree/data/ontology/DocREDOntology/ontology.ttl",
        PROJECT_ROOT.parent / "ragtree/data/ontology/DocREDOntology/ontology.ttl",
        PROJECT_ROOT.parent / "RAGTree/data/ontology/DocREDOntology/ontology.ttl",
        PROJECT_ROOT / "ragtree/data/ontology/DocREDOntology/ontology.ttl",
        PROJECT_ROOT / "data/ontology/DocREDOntology/ontology.ttl",
        PROJECT_ROOT / "data/ontology/redocred_ontology.ttl",
    ],
)

USER_GUIDANCE_PATH = first_existing_path(
    "USER_GUIDANCE_PATH",
    [
        PROJECT_ROOT / "examples/RAGTreeDatasets/configs/guidance_docred.json",
        NOTEBOOK_DIR / "configs/guidance_docred.json",
    ],
)

# Benchmark outputs.
RUNS_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets/runs"
OUTPUT_JSONL = RUNS_DIR / "neoolaf_docred_predictions.docred_constrained.canonical.jsonl"
ARTIFACTS_ROOT = RUNS_DIR / "neoolaf_docred_artifacts_docred_constrained"
SUMMARY_OUTPUT = RUNS_DIR / "neoolaf_docred_run_summary.json"
ERROR_LOG_JSONL = RUNS_DIR / "neoolaf_docred_errors.jsonl"
ARTIFACT_INDEX_JSONL = RUNS_DIR / "neoolaf_docred_evaluation_artifact_index.jsonl"

# OpenRouter configuration.
OPENROUTER_HOST = "https://openrouter.ai/api/v1"
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY", "")
MODEL_NAME = "openai/gpt-oss-20b"

# Smoke test first, then set MAX_DOCS=None for the full split.
RUN_NEOOLAF = True
MAX_DOCS = 5
DOCUMENT_WORKERS = 2
MAX_WORKERS = 2
DISABLE_WIKIPEDIA = False  # runner flags below are enough; no sitecustomize needed
OFFLINE_ONTOLOGY_ONLY = True  # disables web enrichment and blocks Wikipedia/Wikimedia

# Controlled DocRED relation vocabulary.
# For DocRED, dataset is safest because the ontology may contain entity classes only.
# This extracts only the set of relation IDs/labels from the dataset keys, never gold pairs.
RELATION_FILTER_MODE = "list-only"
RELATION_VOCAB_SOURCE = "dataset"  # auto | dataset | ontology | json | union
FORCE_RELATION_VOCABULARY = True
SOURCE_ENTITY_ANCHORING = True

# Direct DocRED-constrained extraction.
# This adds one benchmark-facing extraction call per document that outputs
# head_id, relation_id, tail_id, evidence directly. Native NeoOLAF KG/ontology
# artifacts are still produced by the full pipeline and are not overwritten.
DOCRED_DIRECT_CONSTRAINED_EXTRACTION = True
DOCRED_DIRECT_OUTPUT_MODE = "replace"  # replace | supplement
DOCRED_DIRECT_MAX_ENTITIES = None
DOCRED_DIRECT_MAX_RELATIONS = None
DOCRED_DIRECT_TEMPERATURE = 0.0
DOCRED_DIRECT_RETRIES = 3
DOCRED_DIRECT_RETRY_SLEEP = 2.0
DOCRED_DIRECT_FALLBACK_ON_ERROR = "native"  # native | empty | fail
DOCRED_DIRECT_DISABLE_HINTS = False

# Gold-free DocRED relation calibration and recall recovery.
DOCRED_RELATION_FAMILY_FILTER = True
DOCRED_CALIBRATE_RELATIONS = True
DOCRED_VERIFICATION_PASS = True
DOCRED_ZERO_RELATION_FAMILY_PROBES = True
DOCRED_ZERO_RELATION_PROBE_MAX_FAMILIES = 3
DOCRED_STRICT_TYPE_CONSTRAINTS = True

# Strict DocRED scoring calibration and gold-free closure rules.
# These are post-extraction deterministic rules for benchmark scoring:
# reject weak false positives, relabel common confusions, and add closure facts.
DOCRED_SCORING_CALIBRATION = True
DOCRED_REJECT_PERIPHERAL_RELATIONS = True
DOCRED_GEO_COUNTRY_CLOSURE = True
DOCRED_ORG_CLOSURE = True
DOCRED_DATE_CLOSURE = True
DOCRED_NATIONALITY_CLOSURE = True

# Optional, explicit relation subset for ablation/debugging.
# Keep None for the full DocRED vocabulary. For the 5-doc smoke only, one may test:
# "P17,P27,P131,P150,P69,P159,P749,P30,P264,P577,P127,P571,P355,P108,P361,P527,P175,P19,P569,P570,P495"
DOCRED_DIRECT_FOCUS_RELATION_IDS = None

# Avoid document-level gold leakage during smoke tests. The direct extractor
# receives only the global relation vocabulary, source entities, and text.
FEW_SHOT_FROM_DATASET = False
FEW_SHOT_K = 0

REFERENCE_ONTOLOGY_PATH = NEOOLAF_ONTOLOGY_PATH
RELATION_VOCAB_PATH = None
SOURCE_RELATION_VOCAB_PATH = DATASET_JSONL
RELATION_VOCAB_OUTPUT_PATH = RUNS_DIR / "docred_allowed_relations.json"

# Add extra runner flags here. NeoOLAF source is not changed.
EXTRA_RUNNER_ARGS: list[str] = []

RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"RUNS_DIR={RUNS_DIR}")
print(f"OUTPUT_JSONL={OUTPUT_JSONL}")
print(f"ARTIFACTS_ROOT={ARTIFACTS_ROOT}")
print(f"SUMMARY_OUTPUT={SUMMARY_OUTPUT}")
print(f"ERROR_LOG_JSONL={ERROR_LOG_JSONL}")
print(f"RELATION_VOCAB_OUTPUT_PATH={RELATION_VOCAB_OUTPUT_PATH}")

if RUN_NEOOLAF and not OPENROUTER_KEY:
    raise RuntimeError("Set OPENROUTER_API_KEY before running NeoOLAF.")


DATASET_JSONL=/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal.jsonl
NEOOLAF_ONTOLOGY_PATH=/home/galencarmedeiro/git/postdoc/ragtree/data/ontology/DocREDOntology/ontology.ttl
USER_GUIDANCE_PATH=/home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/guidance_docred.json
RUNS_DIR=/home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs
OUTPUT_JSONL=/home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/neoolaf_docred_predictions.docred_constrained.canonical.jsonl
ARTIFACTS_ROOT=/home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/neoolaf_docred_artifacts_docred_constrained
SUMMARY_OUTPUT=/home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/neoolaf_docred_run_summary.json
ERROR_LOG_JSONL=/home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/neoolaf_docred_errors.jsonl
RELATION_VOCAB_OUTPUT_PATH=/home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatase

## Run NeoOLAF

In [ ]:
command = [
    sys.executable,
    str(RUNNER_PATH),
    "--dataset-jsonl-path", str(DATASET_JSONL),
    "--ontology-path", str(NEOOLAF_ONTOLOGY_PATH),
    "--output-jsonl-path", str(OUTPUT_JSONL),
    "--backend-name", "openrouter",
    "--host", OPENROUTER_HOST,
    "--api-key", OPENROUTER_KEY,
    "--model-name", MODEL_NAME,
    "--type-filter", "dev",
    "--user-guidance-path", str(USER_GUIDANCE_PATH),
    "--relation-vocab-source", RELATION_VOCAB_SOURCE,
    "--relation-vocab-dataset-path", str(DATASET_JSONL),
    "--relation-vocab-ontology-path", str(NEOOLAF_ONTOLOGY_PATH),
    "--relation-vocab-output-path", str(RELATION_VOCAB_OUTPUT_PATH),
    "--force-relation-vocabulary",
    "--source-entity-anchoring",
    "--docred-direct-constrained-extraction",
    "--docred-direct-output-mode", DOCRED_DIRECT_OUTPUT_MODE,
    "--docred-direct-temperature", str(DOCRED_DIRECT_TEMPERATURE),
    "--docred-direct-retries", str(DOCRED_DIRECT_RETRIES),
    "--docred-direct-retry-sleep", str(DOCRED_DIRECT_RETRY_SLEEP),
    "--docred-direct-fallback-on-error", DOCRED_DIRECT_FALLBACK_ON_ERROR,
    "--output-format", "canonical",
    "--artifacts-root", str(ARTIFACTS_ROOT),
    "--chunk-size", "10000000",
    "--chunk-overlap", "0",
    "--max-chunks", "1",
    "--max-expressions", "20",
    "--max-relation-mentions", "20",
    "--max-workers", str(MAX_WORKERS),
    "--document-workers", str(DOCUMENT_WORKERS),
    "--max-tokens", "8192",
    "--openrouter-reasoning-effort", "minimal",
    "--openrouter-exclude-reasoning",
    "--error-log-jsonl-path", str(ERROR_LOG_JSONL),
    "--summary-output-path", str(SUMMARY_OUTPUT),
    "--no-web-search",
    "--disable-wikipedia-lookups",
    "--offline-ontology-only",
    "--no-checkpoints",
    "--no-chunk-checkpoints",
    "--no-resume",
]

if FEW_SHOT_FROM_DATASET:
    command.extend([
        "--few-shot-from-dataset",
        "--few-shot-source-type", "dev",
        "--few-shot-k", str(FEW_SHOT_K),
    ])
if not DOCRED_DIRECT_CONSTRAINED_EXTRACTION:
    # Remove the direct extractor flags if disabled in the config cell.
    cleaned = []
    skip_next = 0
    flags_with_values = {
        "--docred-direct-output-mode",
        "--docred-direct-temperature",
        "--docred-direct-retries",
        "--docred-direct-retry-sleep",
        "--docred-direct-fallback-on-error",
    }
    flags_no_values = {"--docred-direct-constrained-extraction"}
    i = 0
    while i < len(command):
        if command[i] in flags_no_values:
            i += 1
            continue
        if command[i] in flags_with_values:
            i += 2
            continue
        cleaned.append(command[i])
        i += 1
    command = cleaned
if DOCRED_DIRECT_MAX_ENTITIES is not None:
    command.extend(["--docred-direct-max-entities", str(DOCRED_DIRECT_MAX_ENTITIES)])
if DOCRED_DIRECT_MAX_RELATIONS is not None:
    command.extend(["--docred-direct-max-relations", str(DOCRED_DIRECT_MAX_RELATIONS)])
if DOCRED_DIRECT_FOCUS_RELATION_IDS:
    command.extend(["--docred-direct-focus-relation-ids", str(DOCRED_DIRECT_FOCUS_RELATION_IDS)])
if DOCRED_DIRECT_DISABLE_HINTS:
    command.append("--docred-direct-disable-hints")
if DOCRED_RELATION_FAMILY_FILTER:
    command.append("--docred-relation-family-filter")
if DOCRED_CALIBRATE_RELATIONS:
    command.append("--docred-calibrate-relations")
if DOCRED_VERIFICATION_PASS:
    command.append("--docred-verification-pass")
if DOCRED_ZERO_RELATION_FAMILY_PROBES:
    command.append("--docred-zero-relation-family-probes")
    command.extend(["--docred-zero-relation-probe-max-families", str(DOCRED_ZERO_RELATION_PROBE_MAX_FAMILIES)])
if DOCRED_STRICT_TYPE_CONSTRAINTS:
    command.append("--docred-strict-type-constraints")
if DOCRED_SCORING_CALIBRATION:
    command.append("--docred-scoring-calibration")
if DOCRED_REJECT_PERIPHERAL_RELATIONS:
    command.append("--docred-reject-peripheral-relations")
if DOCRED_GEO_COUNTRY_CLOSURE:
    command.append("--docred-geo-country-closure")
if DOCRED_ORG_CLOSURE:
    command.append("--docred-org-closure")
if DOCRED_DATE_CLOSURE:
    command.append("--docred-date-closure")
if DOCRED_NATIONALITY_CLOSURE:
    command.append("--docred-nationality-closure")
if MAX_DOCS is not None:
    command.extend(["--max-docs", str(MAX_DOCS)])
command.extend(EXTRA_RUNNER_ARGS)

print("NeoOLAF command:")
print(shlex.join(command))

if RUN_NEOOLAF:
    runner_env = os.environ.copy()
    if OFFLINE_ONTOLOGY_ONLY:
        print("[NeoOLAF benchmark] OFFLINE_ONTOLOGY_ONLY=True (--no-web-search, --disable-wikipedia-lookups)")
    if DISABLE_WIKIPEDIA:
        if not (OFFLINE_RUNTIME_DIR / "sitecustomize.py").is_file():
            raise FileNotFoundError(
                f"Offline runtime policy not found: {OFFLINE_RUNTIME_DIR}"
            )
        current_pythonpath = runner_env.get("PYTHONPATH", "")
        runner_env["PYTHONPATH"] = os.pathsep.join(
            value
            for value in (str(OFFLINE_RUNTIME_DIR), current_pythonpath)
            if value
        )
        runner_env["NEOOLAF_DISABLE_WIKIPEDIA"] = "1"
        print("[NeoOLAF benchmark] DISABLE_WIKIPEDIA=True")

    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        env=runner_env,
        check=False,
    )
    if completed.returncode != 0:
        print(f"[NeoOLAF benchmark][runner-error] returncode={completed.returncode}")
        print(f"[NeoOLAF benchmark][runner-error] summary={SUMMARY_OUTPUT}")
        print(f"[NeoOLAF benchmark][runner-error] error_log={ERROR_LOG_JSONL}")
        if SUMMARY_OUTPUT.is_file():
            print(SUMMARY_OUTPUT.read_text(encoding="utf-8")[:3000])
        if ERROR_LOG_JSONL.is_file():
            print(ERROR_LOG_JSONL.read_text(encoding="utf-8")[:3000])
        raise RuntimeError(
            "NeoOLAF runner failed. See the printed summary/error_log paths above. "
            "If Wikipedia rate limits appear, make sure the copied run_neoolaf.py is the patched version "
            "and the command contains --no-web-search --disable-wikipedia-lookups --offline-ontology-only."
        )
else:
    print("RUN_NEOOLAF=False, using the existing prediction JSONL and artifacts.")

NeoOLAF command:
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/bin/python /home/galencarmedeiro/git/postdoc/NeoOLAF/experiments/methods/run_neoolaf.py --dataset-jsonl-path /home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal.jsonl --ontology-path /home/galencarmedeiro/git/postdoc/ragtree/data/ontology/DocREDOntology/ontology.ttl --output-jsonl-path /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/neoolaf_docred_predictions.docred_constrained.canonical.jsonl --backend-name openrouter --host https://openrouter.ai/api/v1 --api-key REPLACE_WITH_OPENROUTER_API_KEY_FROM_ENV --model-name openai/gpt-oss-20b --type-filter dev --user-guidance-path /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/guidance_docred.json --relation-vocab-source dataset --relation-vocab-dataset-path /home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal.jsonl --relation-vocab-ontology-path /home/galencarmedeiro/git/postdoc/ragtr

/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


[NeoOLAF benchmark] Wikipedia/Wikidata enrichment disabled by offline source objects.
[NeoOLAF benchmark] Web-search enrichment disabled (--no-web-search).


## Create Controlled Evaluation KGs

This stage reads native NeoOLAF artifacts only. The source document
supplies canonical entities, while the configured relation vocabulary
controls which predicates may appear in evaluation.

In [ ]:
def iter_jsonl(path: Path):
    # Stream records to avoid loading a full benchmark file into memory.
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                yield json.loads(line)


def resolve_artifact_dir(raw_path: str) -> Path:
    # Runner outputs may store absolute paths or paths relative to the repo.
    path = Path(raw_path)
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()


def find_native_kg_exports(artifact_dir: Path) -> tuple[Path, Path]:
    # Prefer local/inferred exports that share the same directory.
    local_candidates = sorted(
        artifact_dir.rglob("kg_local.json"),
        key=lambda path: ("data/exports" not in path.as_posix(), len(path.parts)),
    )
    for local_path in local_candidates:
        inferred_path = local_path.with_name("kg_inferred.json")
        if inferred_path.is_file():
            return local_path, inferred_path
    raise FileNotFoundError(
        f"No kg_local.json + kg_inferred.json pair found under {artifact_dir}"
    )


def projection_command(
    document_id: str,
    kg_local: Path,
    kg_inferred: Path,
    output_dir: Path,
) -> list[str]:
    # Build the independent post-processing command.
    command = [
        sys.executable,
        str(PROJECTOR_PATH),
        "--kg-local-json", str(kg_local),
        "--kg-inferred-json", str(kg_inferred),
        "--source-doc-json", str(DATASET_JSONL),
        "--document-id", document_id,
        "--relation-filter-mode", RELATION_FILTER_MODE,
        "--output-dir", str(output_dir),
    ]
    if REFERENCE_ONTOLOGY_PATH is not None:
        command.extend(["--ontology-path", str(REFERENCE_ONTOLOGY_PATH)])
    if RELATION_VOCAB_PATH is not None:
        command.extend(["--relation-vocab-json", str(RELATION_VOCAB_PATH)])
    if SOURCE_RELATION_VOCAB_PATH is not None:
        command.extend(
            ["--source-relation-vocab-json", str(SOURCE_RELATION_VOCAB_PATH)]
        )
    return command

In [ ]:
if not OUTPUT_JSONL.is_file():
    raise FileNotFoundError(f"Prediction file not found: {OUTPUT_JSONL}")
if not PROJECTOR_PATH.is_file():
    raise FileNotFoundError(f"Evaluation projector not found: {PROJECTOR_PATH}")

artifact_index = []
projection_errors = []

for prediction_record in iter_jsonl(OUTPUT_JSONL):
    if not prediction_record.get("parsed_ok", False):
        continue

    document_id = str(prediction_record["document_id"])
    artifact_dir = resolve_artifact_dir(prediction_record["artifact_dir"])

    try:
        kg_local, kg_inferred = find_native_kg_exports(artifact_dir)
        exports_dir = kg_local.parent
        command = projection_command(
            document_id=document_id,
            kg_local=kg_local,
            kg_inferred=kg_inferred,
            output_dir=exports_dir,
        )
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)

        index_record = {
            "document_id": document_id,
            "artifact_dir": str(artifact_dir),
            "exports_dir": str(exports_dir),
            "kg_local": str(kg_local),
            "kg_inferred": str(kg_inferred),
            "evaluation_kg_local": str(exports_dir / "evaluation_kg_local.json"),
            "evaluation_kg_inferred": str(exports_dir / "evaluation_kg_inferred.json"),
            "evaluation_kg_combined": str(exports_dir / "evaluation_kg_combined.json"),
            "evaluation_projection_report": str(
                exports_dir / "evaluation_projection_report.json"
            ),
            "relation_filter_mode": RELATION_FILTER_MODE,
        }
        artifact_index.append(index_record)

        print(f"[NeoOLAF benchmark][exports] document_id={document_id}")
        for key, value in index_record.items():
            if key not in {"document_id", "relation_filter_mode"}:
                print(f"[NeoOLAF benchmark][exports] {key}={value}")
    except Exception as error:
        projection_errors.append(
            {
                "document_id": document_id,
                "artifact_dir": str(artifact_dir),
                "error_type": type(error).__name__,
                "error": str(error),
            }
        )
        print(
            f"[NeoOLAF benchmark][evaluation-error] document_id={document_id} "
            f"{type(error).__name__}: {error}"
        )

with ARTIFACT_INDEX_JSONL.open("w", encoding="utf-8") as handle:
    for record in artifact_index:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"[NeoOLAF benchmark][exports] artifact_index={ARTIFACT_INDEX_JSONL}")
print(
    f"[NeoOLAF benchmark][exports] projected={len(artifact_index)} "
    f"errors={len(projection_errors)}"
)

if projection_errors:
    error_path = RUNS_DIR / "neoolaf_docred_evaluation_projection_errors.json"
    error_path.write_text(
        json.dumps(projection_errors, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print(f"[NeoOLAF benchmark][exports] projection_errors={error_path}")

## Projection Summary

In [ ]:
summary_rows = []
for record in artifact_index:
    report_path = Path(record["evaluation_projection_report"])
    report = json.loads(report_path.read_text(encoding="utf-8"))
    row = {
        "document_id": record["document_id"],
        "mode": report["relation_filter_mode"],
        "entities": report["entity_count"],
        "allowed_relations": report["allowed_relation_count"],
    }
    for source_report in report["reports"]:
        source = source_report["source_name"]
        row[f"{source}_entities"] = source_report["projected_entities"]
        row[f"{source}_relations"] = source_report["projected_relations"]
        row[f"{source}_accepted"] = source_report["accepted_triples"]
        row[f"{source}_rejected"] = source_report["rejected_triples"]
    summary_rows.append(row)

print(json.dumps(summary_rows, ensure_ascii=False, indent=2))